# WeedyRice: proposed six-channel model, seed 42

Dataset-specific retraining using the official split. The saved source cell has no output; attach the final run exports.

Source: `gd-second.ipynb`, zero-based cell 8.

Run in a fresh Kaggle GPU session with the appropriate dataset attached. Review the configuration near the top; original dataset/output paths are retained. Do not Run All across the repository.


In [ ]:

"""
WeedyRice-RGBMS-DB — Proposed 6-channel gated-fusion model
Fresh training on the official train/val/test split.

Input channels:
    RGB: R, G, B
    MS : Green, Red-Edge, NIR
The dataset's MS Red band is intentionally not used in this first controlled
transfer experiment, so the architecture remains exactly 6-channel like the
GD-RGBMS experiment.

FAST_PILOT=True:
    max 30 epochs, patience 6
FAST_PILOT=False:
    max 70 epochs, patience 12
"""

from __future__ import annotations

import csv
import json
import random
import shutil
import time
from pathlib import Path

import numpy as np
from PIL import Image
from tqdm import tqdm

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader


# =============================================================================
# 1. CONFIG
# =============================================================================

ROOT = Path("/kaggle/input/datasets/hemal7/weedy-rgb-ms")

# Automatically find the actual dataset directory containing the split files.
train_list_candidates = list(ROOT.rglob("train_list.txt"))
if not train_list_candidates:
    raise FileNotFoundError(f"Could not find train_list.txt under {ROOT}")

DATA_DIR = train_list_candidates[0].parent

RGB_DIR = DATA_DIR / "RGB"
MS_DIR = DATA_DIR / "Multispectral"
MASK_DIR = DATA_DIR / "Masks"

for p in (RGB_DIR, MS_DIR, MASK_DIR):
    if not p.exists():
        raise FileNotFoundError(f"Missing required folder: {p}")

WORK_DIR = Path("/kaggle/working")
OUT_DIR = WORK_DIR / "weedy_proposed_6ch_seed42"
OUT_DIR.mkdir(parents=True, exist_ok=True)

SEED = 42

# Quick first run.
FAST_PILOT = False

if FAST_PILOT:
    MAX_EPOCHS = 30
    EARLY_STOPPING_PATIENCE = 6
else:
    MAX_EPOCHS = 50
    EARLY_STOPPING_PATIENCE = 12

TARGET_SIZE = (896, 672)  # width, height
BATCH_SIZE = 2
ACCUMULATION_STEPS = 2
LEARNING_RATE = 1e-4
WEIGHT_DECAY = 1e-4
NUM_WORKERS = 2

BRANCH_BASE = 28
PREFERRED_GN_GROUPS = 8
MODALITY_DROPOUT_PROBABILITY = 0.0

RGB_CHANNELS = 3
SPECTRAL_CHANNELS = 3
TOTAL_INPUT_CHANNELS = 6
INPUT_CHANNEL_ORDER = ("R", "G", "B", "Green", "Red-Edge", "NIR")

# Same auxiliary supervision schedule as the GD model.
AUX_RGB_INITIAL_WEIGHT = 0.15
AUX_SPECTRAL_INITIAL_WEIGHT = 0.25
AUX_RGB_MIDDLE_WEIGHT = 0.10
AUX_SPECTRAL_MIDDLE_WEIGHT = 0.15
AUX_RGB_FINAL_WEIGHT = 0.05
AUX_SPECTRAL_FINAL_WEIGHT = 0.08
AUX_MIDDLE_PHASE_START = 0.40
AUX_FINAL_PHASE_START = 0.70

COARSE_THRESHOLDS = np.round(np.arange(0.20, 0.86, 0.05), 2).tolist()
FINE_THRESHOLDS = np.round(np.arange(0.20, 0.801, 0.01), 2).tolist()

CACHE_DATA = True

try:
    BILINEAR = Image.Resampling.BILINEAR
    NEAREST = Image.Resampling.NEAREST
except AttributeError:
    BILINEAR = Image.BILINEAR
    NEAREST = Image.NEAREST


def seed_everything(seed: int):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.benchmark = False
    torch.backends.cudnn.deterministic = True


seed_everything(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
pin_memory = device.type == "cuda"

print("=" * 95)
print("WEEDY RICE — PROPOSED 6-CHANNEL GATED-FUSION MODEL")
print("=" * 95)
print("ROOT        :", ROOT)
print("DATA_DIR    :", DATA_DIR)
print("RGB_DIR     :", RGB_DIR)
print("MS_DIR      :", MS_DIR)
print("MASK_DIR    :", MASK_DIR)
print("Device      :", device)
print("Fast pilot  :", FAST_PILOT)
print("Max epochs  :", MAX_EPOCHS)
print("Patience    :", EARLY_STOPPING_PATIENCE)
print("Input size  :", TARGET_SIZE)
print("Channels    :", INPUT_CHANNEL_ORDER)


# =============================================================================
# 2. SPLIT LISTS
# =============================================================================

def read_list(path: Path):
    with open(path, "r", encoding="utf-8") as f:
        return [line.strip() for line in f if line.strip()]


train_names = read_list(DATA_DIR / "train_list.txt")
val_names = read_list(DATA_DIR / "val_list.txt")
test_names = read_list(DATA_DIR / "test_list.txt")

print("\nOfficial splits:")
print("Train:", len(train_names))
print("Val  :", len(val_names))
print("Test :", len(test_names))
print("Total:", len(train_names) + len(val_names) + len(test_names))

if len(set(train_names) & set(val_names)) > 0:
    raise RuntimeError("Train/val overlap detected.")
if len(set(train_names) & set(test_names)) > 0:
    raise RuntimeError("Train/test overlap detected.")
if len(set(val_names) & set(test_names)) > 0:
    raise RuntimeError("Val/test overlap detected.")


# =============================================================================
# 3. DATA LOADING
# =============================================================================

def filename_stem(name: str) -> str:
    return Path(name).stem


def image_to_unit_float(path: Path, resize_size=None, resample=BILINEAR):
    """
    Reads 8-bit or 16-bit single/multi-channel images and maps integer values
    linearly to [0,1]. For float images, values are kept if already in [0,1];
    otherwise they are scaled by their observed max for that image.
    """
    with Image.open(path) as img:
        if resize_size is not None:
            img = img.resize(resize_size, resample)
        arr = np.asarray(img).copy()

    original_dtype = arr.dtype

    if np.issubdtype(original_dtype, np.integer):
        max_value = np.iinfo(original_dtype).max
        arr = arr.astype(np.float32) / float(max_value)
    else:
        arr = arr.astype(np.float32)
        vmax = float(np.nanmax(arr)) if arr.size else 1.0
        if vmax > 1.0:
            arr = arr / max(vmax, 1e-6)

    return arr


def load_rgb(path: Path):
    with Image.open(path) as img:
        img = img.convert("RGB").resize(TARGET_SIZE, BILINEAR)
        rgb = np.asarray(img, dtype=np.float32).copy() / 255.0
    return rgb


def load_single_band(path: Path):
    arr = image_to_unit_float(path, resize_size=TARGET_SIZE, resample=BILINEAR)

    if arr.ndim == 3:
        arr = arr[..., 0]

    if arr.ndim != 2:
        raise RuntimeError(f"Expected a single-band image, got {arr.shape}: {path}")

    return arr.astype(np.float32)


def load_mask(path: Path):
    with Image.open(path) as img:
        img = img.convert("L").resize(TARGET_SIZE, NEAREST)
        arr = np.asarray(img, dtype=np.uint8)
    # Official masks: 255 = weedy rice, 0 = background
    return (arr > 127).astype(np.float32)


def paths_for(name: str):
    stem = filename_stem(name)

    rgb_path = RGB_DIR / f"{stem}.JPG"
    if not rgb_path.exists():
        rgb_path = RGB_DIR / f"{stem}.jpg"

    g_path = MS_DIR / f"{stem}_G.TIF"
    re_path = MS_DIR / f"{stem}_RE.TIF"
    nir_path = MS_DIR / f"{stem}_NIR.TIF"

    mask_path = MASK_DIR / f"{stem}.png"

    for p in (rgb_path, g_path, re_path, nir_path, mask_path):
        if not p.exists():
            raise FileNotFoundError(f"Missing paired file: {p}")

    return rgb_path, g_path, re_path, nir_path, mask_path


def load_raw(name: str):
    rgb_path, g_path, re_path, nir_path, mask_path = paths_for(name)

    rgb = load_rgb(rgb_path)
    green = load_single_band(g_path)
    red_edge = load_single_band(re_path)
    nir = load_single_band(nir_path)
    mask = load_mask(mask_path)

    ms = np.stack([green, red_edge, nir], axis=-1)
    image = np.concatenate([rgb, ms], axis=-1).astype(np.float32)

    if image.shape != (TARGET_SIZE[1], TARGET_SIZE[0], 6):
        raise RuntimeError(f"Unexpected image shape {image.shape} for {name}")

    return image, mask, filename_stem(name)


# Quick pairing check.
print("\nChecking one sample...")
sample_image, sample_mask, sample_stem = load_raw(train_names[0])
print("Sample stem :", sample_stem)
print("Image shape :", sample_image.shape)
print("Mask shape  :", sample_mask.shape)
print("Image range :", float(sample_image.min()), "to", float(sample_image.max()))
print("Mask values :", np.unique(sample_mask))


# =============================================================================
# 4. OPTIONAL RAM CACHE
# =============================================================================

RAW_CACHE = {}

all_names = train_names + val_names + test_names

if CACHE_DATA:
    print("\nCaching resized 6-channel samples...")
    for name in tqdm(all_names, desc="Cache"):
        image, mask, stem = load_raw(name)
        RAW_CACHE[name] = (
            image.astype(np.float16),
            mask.astype(np.uint8),
            stem,
        )
    total_gb = sum(
        x.nbytes + y.nbytes for x, y, _ in RAW_CACHE.values()
    ) / (1024 ** 3)
    print(f"Cached {len(RAW_CACHE)} samples (~{total_gb:.2f} GB)")


def get_raw(name):
    if name in RAW_CACHE:
        image, mask, stem = RAW_CACHE[name]
        return image.astype(np.float32), mask.astype(np.float32), stem
    return load_raw(name)


# =============================================================================
# 5. TRAIN-ONLY NORMALIZATION
# =============================================================================

def compute_train_stats(names):
    channel_sum = np.zeros(TOTAL_INPUT_CHANNELS, dtype=np.float64)
    channel_sq = np.zeros(TOTAL_INPUT_CHANNELS, dtype=np.float64)
    count = 0

    for name in tqdm(names, desc="Train normalization"):
        image, _, _ = get_raw(name)

        channel_sum += image.sum(axis=(0, 1), dtype=np.float64)
        channel_sq += np.square(image, dtype=np.float32).sum(
            axis=(0, 1), dtype=np.float64
        )
        count += image.shape[0] * image.shape[1]

    mean64 = channel_sum / count
    var64 = channel_sq / count - np.square(mean64)

    mean = mean64.astype(np.float32)
    std = np.sqrt(np.maximum(var64, 1e-12)).astype(np.float32)

    return mean, std


MEAN, STD = compute_train_stats(train_names)

print("\nTraining-set normalization:")
for name, mean, std in zip(INPUT_CHANNEL_ORDER, MEAN, STD):
    print(f"{name:10s} mean={mean:.6f} std={std:.6f}")

with open(OUT_DIR / "normalization.json", "w", encoding="utf-8") as f:
    json.dump({
        "channel_order": list(INPUT_CHANNEL_ORDER),
        "mean": MEAN.tolist(),
        "std": STD.tolist(),
        "source": "official training split only",
    }, f, indent=2)


# =============================================================================
# 6. DATASET CLASS
# =============================================================================

class WeedyRiceDataset(Dataset):
    def __init__(self, names, mean, std, augment=False):
        self.names = list(names)
        self.mean = np.asarray(mean, dtype=np.float32).reshape(1, 1, -1)
        self.std = np.asarray(std, dtype=np.float32).reshape(1, 1, -1)
        self.augment = augment

    def __len__(self):
        return len(self.names)

    def __getitem__(self, index):
        name = self.names[index]
        image, mask, stem = get_raw(name)

        if self.augment:
            if random.random() < 0.5:
                image = np.flip(image, axis=1).copy()
                mask = np.flip(mask, axis=1).copy()

            if random.random() < 0.5:
                image = np.flip(image, axis=0).copy()
                mask = np.flip(mask, axis=0).copy()

            # Same visible-image perturbation used for GD.
            if random.random() < 0.75:
                gain = np.random.uniform(
                    0.88, 1.12, size=(1, 1, 3)
                ).astype(np.float32)
                bias = np.random.uniform(
                    -0.035, 0.035, size=(1, 1, 3)
                ).astype(np.float32)
                image[..., :3] = np.clip(
                    image[..., :3] * gain + bias, 0.0, 1.0
                )

        image = ((image - self.mean) / self.std).astype(np.float32)

        return (
            torch.from_numpy(image.transpose(2, 0, 1)).float(),
            torch.from_numpy(mask[None]).float(),
            stem,
        )


def make_loader(names, augment=False, batch_size=BATCH_SIZE, shuffle=False):
    return DataLoader(
        WeedyRiceDataset(names, MEAN, STD, augment=augment),
        batch_size=batch_size,
        shuffle=shuffle,
        num_workers=NUM_WORKERS,
        pin_memory=pin_memory,
        drop_last=False,
    )


train_loader = make_loader(train_names, augment=True, batch_size=BATCH_SIZE, shuffle=True)
val_loader = make_loader(val_names, augment=False, batch_size=1, shuffle=False)
test_loader = make_loader(test_names, augment=False, batch_size=1, shuffle=False)


# =============================================================================
# 7. EXACT PROPOSED MODEL
# =============================================================================

def group_norm(channels):
    groups = min(PREFERRED_GN_GROUPS, channels)
    while channels % groups:
        groups -= 1
    return nn.GroupNorm(groups, channels, eps=1e-5, affine=True)


class SCSE(nn.Module):
    def __init__(self, channels, reduction=16):
        super().__init__()
        hidden = max(channels // reduction, 1)

        self.channel = nn.Sequential(
            nn.AdaptiveAvgPool2d(1),
            nn.Conv2d(channels, hidden, 1),
            nn.SiLU(inplace=True),
            nn.Conv2d(hidden, channels, 1),
            nn.Sigmoid(),
        )

        self.spatial = nn.Sequential(
            nn.Conv2d(channels, 1, 1),
            nn.Sigmoid(),
        )

    def forward(self, x):
        return x * self.channel(x) + x * self.spatial(x)


class ResBlock(nn.Module):
    def __init__(self, in_channels, out_channels, dropout=0.0):
        super().__init__()

        self.block = nn.Sequential(
            nn.Conv2d(in_channels, out_channels, 3, padding=1, bias=False),
            group_norm(out_channels),
            nn.SiLU(inplace=True),
            nn.Dropout2d(dropout) if dropout else nn.Identity(),
            nn.Conv2d(out_channels, out_channels, 3, padding=1, bias=False),
            group_norm(out_channels),
        )

        self.skip = (
            nn.Identity()
            if in_channels == out_channels
            else nn.Sequential(
                nn.Conv2d(in_channels, out_channels, 1, bias=False),
                group_norm(out_channels),
            )
        )

        self.attention = SCSE(out_channels)

    def forward(self, x):
        y = self.block(x) + self.skip(x)
        return self.attention(F.silu(y, inplace=True))


class SoftGatedFusion(nn.Module):
    def __init__(self, channels):
        super().__init__()
        hidden = max(channels // 8, 8)

        self.gate = nn.Sequential(
            nn.Conv2d(channels * 2, hidden, 1, bias=False),
            group_norm(hidden),
            nn.SiLU(inplace=True),
            nn.Conv2d(hidden, 2, 1),
        )

        self.refine = nn.Sequential(
            nn.Conv2d(channels, channels, 3, padding=1, bias=False),
            group_norm(channels),
            nn.SiLU(inplace=True),
        )

    def forward(self, rgb_feature, spectral_feature):
        weights = torch.softmax(
            self.gate(torch.cat([rgb_feature, spectral_feature], dim=1)),
            dim=1,
        )

        fused = (
            weights[:, 0:1] * rgb_feature
            + weights[:, 1:2] * spectral_feature
        )

        return self.refine(fused), weights


class UpBlock(nn.Module):
    def __init__(self, in_channels, skip_channels, out_channels, dropout=0.0):
        super().__init__()
        self.conv = ResBlock(
            in_channels + skip_channels,
            out_channels,
            dropout,
        )

    def forward(self, x, skip):
        x = F.interpolate(
            x,
            size=skip.shape[-2:],
            mode="bilinear",
            align_corners=False,
        )
        return self.conv(torch.cat([x, skip], dim=1))


class AuxiliarySupervisedGatedUNetNoNDRE(nn.Module):
    def __init__(self, base=28):
        super().__init__()

        c1, c2, c3, c4 = [
            base,
            base * 2,
            base * 4,
            base * 8,
        ]

        self.pool = nn.MaxPool2d(2)

        self.rgb1 = ResBlock(RGB_CHANNELS, c1)
        self.spec1 = ResBlock(SPECTRAL_CHANNELS, c1)

        self.rgb2 = ResBlock(c1, c2)
        self.spec2 = ResBlock(c1, c2)

        self.rgb3 = ResBlock(c2, c3, 0.05)
        self.spec3 = ResBlock(c2, c3, 0.05)

        self.rgb4 = ResBlock(c3, c4, 0.10)
        self.spec4 = ResBlock(c3, c4, 0.10)

        self.fuse1 = SoftGatedFusion(c1)
        self.fuse2 = SoftGatedFusion(c2)
        self.fuse3 = SoftGatedFusion(c3)
        self.fuse4 = SoftGatedFusion(c4)

        self.rgb_auxiliary_head = nn.Conv2d(c4, 1, 1)
        self.spectral_auxiliary_head = nn.Conv2d(c4, 1, 1)

        self.center = ResBlock(c4, c4 * 2, 0.10)

        self.up4 = UpBlock(c4 * 2, c4, c4, 0.10)
        self.up3 = UpBlock(c4, c3, c3, 0.05)
        self.up2 = UpBlock(c3, c2, c2)
        self.up1 = UpBlock(c2, c1, c1)

        self.output = nn.Conv2d(c1, 1, 1)

    def modality_dropout(self, rgb, spectral):
        if (
            not self.training
            or MODALITY_DROPOUT_PROBABILITY <= 0
        ):
            return rgb, spectral

        draw = random.random()

        if draw < MODALITY_DROPOUT_PROBABILITY / 2:
            rgb = torch.zeros_like(rgb)
        elif draw < MODALITY_DROPOUT_PROBABILITY:
            spectral = torch.zeros_like(spectral)

        return rgb, spectral

    def forward(self, x, return_aux=False):
        if x.ndim != 4 or x.shape[1] != 6:
            raise ValueError(
                f"Expected [B,6,H,W], got {tuple(x.shape)}"
            )

        input_size = x.shape[-2:]

        rgb, spectral = self.modality_dropout(
            x[:, :3],
            x[:, 3:],
        )

        r1 = self.rgb1(rgb)
        s1 = self.spec1(spectral)
        f1, _ = self.fuse1(r1, s1)

        r2 = self.rgb2(self.pool(r1 + f1))
        s2 = self.spec2(self.pool(s1 + f1))
        f2, _ = self.fuse2(r2, s2)

        r3 = self.rgb3(self.pool(r2 + f2))
        s3 = self.spec3(self.pool(s2 + f2))
        f3, _ = self.fuse3(r3, s3)

        r4 = self.rgb4(self.pool(r3 + f3))
        s4 = self.spec4(self.pool(s3 + f3))
        f4, _ = self.fuse4(r4, s4)

        center = self.center(self.pool(f4))

        d4 = self.up4(center, f4)
        d3 = self.up3(d4, f3)
        d2 = self.up2(d3, f2)
        d1 = self.up1(d2, f1)

        logits = self.output(d1)

        if return_aux:
            rgb_aux = F.interpolate(
                self.rgb_auxiliary_head(r4),
                size=input_size,
                mode="bilinear",
                align_corners=False,
            )

            spec_aux = F.interpolate(
                self.spectral_auxiliary_head(s4),
                size=input_size,
                mode="bilinear",
                align_corners=False,
            )

            return logits, rgb_aux, spec_aux

        return logits


def build_model():
    model = AuxiliarySupervisedGatedUNetNoNDRE(BRANCH_BASE).to(device)

    total = sum(p.numel() for p in model.parameters())
    aux = sum(
        p.numel()
        for name, p in model.named_parameters()
        if "auxiliary_head" in name
    )
    deploy = total - aux

    print(
        f"Parameters: total={total:,}, "
        f"auxiliary={aux:,}, deployment={deploy:,}"
    )

    return model, total, aux, deploy


# =============================================================================
# 8. LOSS / METRICS
# =============================================================================

class ComboLoss(nn.Module):
    def forward(self, logits, targets):
        bce_map = F.binary_cross_entropy_with_logits(
            logits,
            targets.float(),
            reduction="none",
        )
        bce = bce_map.mean()

        probabilities = torch.sigmoid(logits)

        pt = (
            probabilities * targets
            + (1 - probabilities) * (1 - targets)
        )
        focal = (
            bce_map * ((1 - pt) ** 2.0)
        ).mean()

        dims = tuple(range(1, logits.ndim))

        intersection = (
            probabilities * targets
        ).sum(dims)

        denominator = (
            probabilities.sum(dims)
            + targets.sum(dims)
        )

        dice = 1 - (
            (2 * intersection + 1e-6)
            / (denominator + 1e-6)
        ).mean()

        return (
            0.35 * bce
            + 0.25 * focal
            + 0.40 * dice
        )


criterion = ComboLoss()


def auxiliary_loss_weights(epoch):
    progress = epoch / max(MAX_EPOCHS, 1)

    if progress <= AUX_MIDDLE_PHASE_START:
        return (
            AUX_RGB_INITIAL_WEIGHT,
            AUX_SPECTRAL_INITIAL_WEIGHT,
        )

    if progress <= AUX_FINAL_PHASE_START:
        return (
            AUX_RGB_MIDDLE_WEIGHT,
            AUX_SPECTRAL_MIDDLE_WEIGHT,
        )

    return (
        AUX_RGB_FINAL_WEIGHT,
        AUX_SPECTRAL_FINAL_WEIGHT,
    )


def confusion(probabilities, masks, threshold):
    if isinstance(probabilities, np.ndarray):
        prediction = probabilities >= threshold
        target = masks >= 0.5

        tp = int(np.logical_and(prediction, target).sum())
        fp = int(np.logical_and(prediction, ~target).sum())
        fn = int(np.logical_and(~prediction, target).sum())
        tn = int(np.logical_and(~prediction, ~target).sum())

        return [tp, fp, fn, tn]

    prediction = probabilities >= threshold
    target = masks >= 0.5

    tp = int((prediction & target).sum().item())
    fp = int((prediction & ~target).sum().item())
    fn = int((~prediction & target).sum().item())
    tn = int((~prediction & ~target).sum().item())

    return [tp, fp, fn, tn]


def metrics(values):
    tp, fp, fn, tn = values
    eps = 1e-7

    return {
        "dice": (2 * tp + eps) / (2 * tp + fp + fn + eps),
        "iou": (tp + eps) / (tp + fp + fn + eps),
        "precision": (tp + eps) / (tp + fp + eps),
        "recall": (tp + eps) / (tp + fn + eps),
        "accuracy": (tp + tn + eps) / (tp + fp + fn + tn + eps),
    }


def save_csv(path, rows):
    if not rows:
        return

    with open(path, "w", newline="", encoding="utf-8") as f:
        writer = csv.DictWriter(
            f,
            fieldnames=list(rows[0].keys()),
        )
        writer.writeheader()
        writer.writerows(rows)


# =============================================================================
# 9. TRAIN / VALIDATE
# =============================================================================

def train_epoch(loader, epoch, model, optimizer, scaler):
    model.train()
    optimizer.zero_grad(set_to_none=True)

    rgb_weight, spectral_weight = auxiliary_loss_weights(epoch)

    sums = {
        "total": 0.0,
        "main": 0.0,
        "rgb_aux": 0.0,
        "spec_aux": 0.0,
    }

    for step, (images, masks, _) in enumerate(
        tqdm(loader, desc=f"Train epoch {epoch}", leave=False),
        start=1,
    ):
        images = images.to(device, non_blocking=True)
        masks = masks.to(device, non_blocking=True)

        with torch.amp.autocast(
            "cuda",
            enabled=device.type == "cuda",
        ):
            main_logits, rgb_aux_logits, spec_aux_logits = model(
                images,
                return_aux=True,
            )

            main_loss = criterion(main_logits, masks)
            rgb_aux_loss = criterion(rgb_aux_logits, masks)
            spec_aux_loss = criterion(spec_aux_logits, masks)

            total_loss = (
                main_loss
                + rgb_weight * rgb_aux_loss
                + spectral_weight * spec_aux_loss
            )

            scaled_loss = (
                total_loss / ACCUMULATION_STEPS
            )

        scaler.scale(scaled_loss).backward()

        sums["total"] += total_loss.item()
        sums["main"] += main_loss.item()
        sums["rgb_aux"] += rgb_aux_loss.item()
        sums["spec_aux"] += spec_aux_loss.item()

        if (
            step % ACCUMULATION_STEPS == 0
            or step == len(loader)
        ):
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(
                model.parameters(),
                1.0,
            )

            scaler.step(optimizer)
            scaler.update()
            optimizer.zero_grad(set_to_none=True)

    return {
        key: value / len(loader)
        for key, value in sums.items()
    }


@torch.no_grad()
def validate_model(loader, model):
    model.eval()

    totals = {
        t: [0, 0, 0, 0]
        for t in COARSE_THRESHOLDS
    }

    loss_total = 0.0

    for images, masks, _ in tqdm(
        loader,
        desc="Validation",
        leave=False,
    ):
        images = images.to(device, non_blocking=True)
        masks = masks.to(device, non_blocking=True)

        with torch.amp.autocast(
            "cuda",
            enabled=device.type == "cuda",
        ):
            logits = model(images)
            loss_total += criterion(logits, masks).item()

        probabilities = torch.sigmoid(logits)

        for threshold in COARSE_THRESHOLDS:
            values = confusion(
                probabilities,
                masks,
                threshold,
            )
            totals[threshold] = [
                a + b
                for a, b in zip(
                    totals[threshold],
                    values,
                )
            ]

    rows = []

    for threshold, values in totals.items():
        row = metrics(values)
        row["threshold"] = float(threshold)
        row["loss"] = loss_total / len(loader)
        rows.append(row)

    return max(rows, key=lambda row: row["dice"])


@torch.no_grad()
def collect_probabilities(loader, model, description):
    model.eval()
    records = []

    for images, masks, stems in tqdm(
        loader,
        desc=description,
        leave=False,
    ):
        images = images.to(device, non_blocking=True)

        with torch.amp.autocast(
            "cuda",
            enabled=device.type == "cuda",
        ):
            probabilities = torch.sigmoid(
                model(images)
            )

        records.append({
            "stem": stems[0],
            "mask": masks[0, 0].numpy().astype(np.float32),
            "probability": probabilities[
                0, 0
            ].cpu().numpy().astype(np.float32),
        })

    return records


def threshold_search(records, thresholds):
    rows = []

    for threshold in thresholds:
        total = [0, 0, 0, 0]

        for record in records:
            values = confusion(
                record["probability"],
                record["mask"],
                threshold,
            )

            total = [
                a + b
                for a, b in zip(total, values)
            ]

        row = metrics(total)
        row["threshold"] = float(threshold)
        rows.append(row)

    return max(
        rows,
        key=lambda row: row["dice"],
    ), rows


# =============================================================================
# 10. TRAIN MODEL
# =============================================================================

seed_everything(SEED)

model, parameter_count, auxiliary_parameter_count, deployment_parameter_count = build_model()

optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=LEARNING_RATE,
    weight_decay=WEIGHT_DECAY,
)

scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer,
    mode="max",
    factor=0.5,
    patience=4,
    min_lr=1e-6,
)

scaler = torch.amp.GradScaler(
    "cuda",
    enabled=device.type == "cuda",
)

best_checkpoint_path = OUT_DIR / "best_checkpoint.pt"

history = []

best_val_dice = -1.0
best_epoch = 0

print("\n" + "=" * 95)
print("TRAINING")
print("=" * 95)

for epoch in range(1, MAX_EPOCHS + 1):
    start = time.time()

    lr_used = optimizer.param_groups[0]["lr"]

    train_result = train_epoch(
        train_loader,
        epoch,
        model,
        optimizer,
        scaler,
    )

    val_result = validate_model(
        val_loader,
        model,
    )

    scheduler.step(val_result["dice"])

    row = {
        "epoch": epoch,
        "learning_rate": lr_used,
        "train_total_loss": train_result["total"],
        "train_main_loss": train_result["main"],
        "train_rgb_aux_loss": train_result["rgb_aux"],
        "train_spec_aux_loss": train_result["spec_aux"],
        "val_loss": val_result["loss"],
        "val_dice": val_result["dice"],
        "val_iou": val_result["iou"],
        "val_precision": val_result["precision"],
        "val_recall": val_result["recall"],
        "val_threshold_coarse": val_result["threshold"],
        "minutes": (time.time() - start) / 60.0,
    }

    history.append(row)

    print(
        f"Epoch {epoch:02d}/{MAX_EPOCHS} | "
        f"loss={train_result['total']:.4f} | "
        f"val Dice={val_result['dice']:.4f} | "
        f"IoU={val_result['iou']:.4f} | "
        f"P={val_result['precision']:.4f} | "
        f"R={val_result['recall']:.4f} | "
        f"th={val_result['threshold']:.2f} | "
        f"lr={lr_used:.2e}"
    )

    if val_result["dice"] > best_val_dice:
        best_val_dice = val_result["dice"]
        best_epoch = epoch

        torch.save({
            "model": model.state_dict(),
            "epoch": epoch,
            "val_dice": best_val_dice,
            "mean": MEAN.tolist(),
            "std": STD.tolist(),
            "channel_order": list(INPUT_CHANNEL_ORDER),
        }, best_checkpoint_path)

    if epoch - best_epoch >= EARLY_STOPPING_PATIENCE:
        print(
            f"\nEarly stopping: no validation improvement "
            f"for {EARLY_STOPPING_PATIENCE} epochs."
        )
        break

save_csv(
    OUT_DIR / "training_history.csv",
    history,
)


# =============================================================================
# 11. FINE THRESHOLD SEARCH ON OFFICIAL VALIDATION SET
# =============================================================================

try:
    checkpoint = torch.load(
        best_checkpoint_path,
        map_location=device,
        weights_only=False,
    )
except TypeError:
    checkpoint = torch.load(
        best_checkpoint_path,
        map_location=device,
    )

model.load_state_dict(
    checkpoint["model"],
    strict=True,
)

val_records = collect_probabilities(
    val_loader,
    model,
    "Fine threshold search on validation",
)

val_best, threshold_rows = threshold_search(
    val_records,
    FINE_THRESHOLDS,
)

selected_threshold = float(
    val_best["threshold"]
)

save_csv(
    OUT_DIR / "validation_threshold_search.csv",
    threshold_rows,
)

print("\n" + "=" * 95)
print("VALIDATION SELECTION")
print("=" * 95)
print("Best epoch        :", best_epoch)
print("Best val Dice     :", f"{best_val_dice:.4f}")
print("Selected threshold:", f"{selected_threshold:.2f}")
print("Fine val Dice     :", f"{val_best['dice']:.4f}")


# =============================================================================
# 12. FINAL TEST — OFFICIAL TEST SET, UNTOUCHED
# =============================================================================

test_records = collect_probabilities(
    test_loader,
    model,
    "Official test evaluation",
)

global_values = [0, 0, 0, 0]
per_image_rows = []

for record in test_records:
    values = confusion(
        record["probability"],
        record["mask"],
        selected_threshold,
    )

    global_values = [
        a + b
        for a, b in zip(
            global_values,
            values,
        )
    ]

    row = metrics(values)

    row.update({
        "stem": record["stem"],
        "threshold": selected_threshold,
        "tp": values[0],
        "fp": values[1],
        "fn": values[2],
        "tn": values[3],
    })

    per_image_rows.append(row)

test_metrics = metrics(global_values)

test_metrics.update({
    "seed": SEED,
    "train_images": len(train_names),
    "val_images": len(val_names),
    "test_images": len(test_names),
    "best_epoch": best_epoch,
    "selected_threshold": selected_threshold,
    "fast_pilot": FAST_PILOT,
    "tp": global_values[0],
    "fp": global_values[1],
    "fn": global_values[2],
    "tn": global_values[3],
})

save_csv(
    OUT_DIR / "per_image_test_metrics.csv",
    per_image_rows,
)

save_csv(
    OUT_DIR / "final_test_metrics.csv",
    [test_metrics],
)

with open(
    OUT_DIR / "experiment_configuration.json",
    "w",
    encoding="utf-8",
) as f:
    json.dump({
        "dataset": "WeedyRice-RGBMS-DB",
        "protocol": "official train/val/test split",
        "seed": SEED,
        "splits": {
            "train": len(train_names),
            "val": len(val_names),
            "test": len(test_names),
        },
        "input_channel_order": list(INPUT_CHANNEL_ORDER),
        "unused_ms_band": "Red",
        "target_size": list(TARGET_SIZE),
        "batch_size": BATCH_SIZE,
        "accumulation_steps": ACCUMULATION_STEPS,
        "max_epochs": MAX_EPOCHS,
        "early_stopping_patience": EARLY_STOPPING_PATIENCE,
        "learning_rate": LEARNING_RATE,
        "weight_decay": WEIGHT_DECAY,
        "best_epoch": best_epoch,
        "selected_threshold": selected_threshold,
        "normalization_mean": MEAN.tolist(),
        "normalization_std": STD.tolist(),
        "parameter_count": parameter_count,
        "deployment_parameter_count": deployment_parameter_count,
        "final_test_metrics": test_metrics,
    }, f, indent=2)


# =============================================================================
# 13. SUMMARY / ZIP
# =============================================================================

print("\n" + "=" * 95)
print("FINAL OFFICIAL TEST RESULT")
print("=" * 95)
print("Train      :", len(train_names))
print("Validation :", len(val_names))
print("Test       :", len(test_names))
print("Best epoch :", best_epoch)
print("Threshold  :", f"{selected_threshold:.2f}")
print("Dice       :", f"{test_metrics['dice']:.4f}")
print("IoU        :", f"{test_metrics['iou']:.4f}")
print("Precision  :", f"{test_metrics['precision']:.4f}")
print("Recall     :", f"{test_metrics['recall']:.4f}")
print("Accuracy   :", f"{test_metrics['accuracy']:.4f}")

archive_path = shutil.make_archive(
    str(WORK_DIR / "weedy_proposed_6ch_seed42_results"),
    "zip",
    str(OUT_DIR),
)

print("\nResults directory:", OUT_DIR)
print("Results ZIP      :", archive_path)
print("\nDone.")
